# Interactive Scenes and Multi-Scene Management

**Part I · Visualization** — Tutorial 06

A single `Visualizer` owns one WebSocket server but can manage **many named
scenes** at once. Each scene is an independent `Scene` with its own entities,
styles, controls, camera, title, and annotation. You will learn to:

- Add to the main scene and create named scenes with `viz.scene("name")`.
- Use `VizSceneHandle` and the scene context managers (`with viz:` /
  `with viz.scene("name"):`).
- Drive the scene lifecycle (`add` / `update` / `remove` / `clear` / `flush`).
- Navigate browsers between scenes and inspect connections.


## Setup


In [1]:
from pytanga.geometry import Point, Sphere
from pytanga.viz import SphereStyle, Visualizer


## 1. The main scene and named scenes

The main scene is named `""` and is served at `/`. Named scenes are created
lazily with `viz.scene("name")` (idempotent) and served at `/<name>`. The call
returns a `VizSceneHandle` that scopes every operation to that scene.


In [2]:
viz = Visualizer(title="Tanga — Multi-Scene")

# The main scene (name "") is targeted by the plain Visualizer API:
viz.add(Point(0, 0, 0), color="#ff4444", label="origin")

# Named scenes:
overview = viz.scene("overview")
detail = viz.scene("detail")

overview.add(Sphere(Point(0, 0, 0), 2.0), color="#4488ff", opacity=0.3)
detail.add(Sphere(Point(2, 1, 0), 1.0), color="#ffcc00", opacity=0.8)

print("scenes:", viz.list_scenes())


scenes: ['', 'overview', 'detail']


## 2. Scene context managers

`Visualizer` and `VizSceneHandle` are context managers: they **clear** the
scene and call `show()` on entry, then `flush()` on exit.


In [3]:
viz = Visualizer(reuse_existing=False, title="Tanga — Multi-Scene")

overview = viz.scene("overview")
detail = viz.scene("detail")

with overview:                       # reset + show this scene, then flush
    overview.set_title("Overview")
    overview.add(Sphere(Point(0, 0, 0), 2.0), color="#4488ff", opacity=0.3)

with detail:                         # reset + show this scene, then flush
    detail.set_title("Detail")
    detail.add(Sphere(Point(2, 1, 0), 1.0), color="#ffcc00", opacity=0.8)

# viz.wait()  # in a script, keep running until Ctrl+C


http://localhost:8765

## 3. Scene lifecycle

Each scene supports `add` / `update` / `remove` / `clear` / `flush`, plus
`update_style()` (change style properties without rebuilding geometry) and
`update_entity()` (replace geometry in place). After a `clear()`, pass
`add_axes=` / `add_grid=` to re-add the default axes/grid.


In [4]:
viz = Visualizer(add_default_axes=False, add_default_grid=False)

sid = viz.add(Sphere(Point(0, 0, 0), 1.5), color="#ffaa00", label="sphere")
print("after add:", viz.list_scenes())

# Change style without rebuilding geometry:
viz.update_style(sid, SphereStyle(wireframe=True, opacity=0.4))

# Replace geometry in place:
viz.update_entity(sid, Sphere(Point(1, 0, 0), 1.5))

# Remove it, then clear the scene (re-adding default axes/grid):
viz.remove(sid)
viz.clear(add_axes=True, add_grid=True)
print("cleared (axes + grid re-added)")
viz.display_snapshot()


after add: ['']
cleared (axes + grid re-added)


## 4. Navigation, browser identity, and inspection

`navigate_to()` redirects connected browsers to another scene, targeting
`"all"`, `"scene:<name>"`, `"browser:<id>"`, or `"viewer:<name>"`. Each
WebSocket connection gets a unique `browser_id`; the `?viewer=` URL parameter
labels a viewer. `list_browsers()` reports the connected browsers (empty when
the server is not running).


In [5]:
viz = Visualizer()
overview = viz.scene("overview")
detail = viz.scene("detail")

# Navigate every browser currently viewing the main scene to "detail":
# viz.navigate_to("detail", target="scene:")

# Or navigate only a specific browser (from a control handler's ControlEvent):
# viz.navigate_to("overview", target=f"browser:{event.browser_id}")

# Inspect connections:
print("scenes:", viz.list_scenes())
print("browsers:", viz.list_browsers())


scenes: ['', 'overview', 'detail']
browsers: []


## 5. Side-by-side Jupyter display — `display_row()`

`display_row()` shows several scenes side-by-side in a single notebook cell.
Each element is a `(handle, viewer_name)` tuple; the `viewer_name` becomes the
`?viewer=` identity used for targeted navigation.


In [6]:
viz = Visualizer()
overview = viz.scene("overview")
detail = viz.scene("detail")

overview.add(Sphere(Point(0, 0, 0), 2.0), opacity=0.3)
detail.add(Sphere(Point(2, 1, 0), 1.0), opacity=0.8)

# Live side-by-side (requires the server; renders inline in Jupyter):
viz.display_row(
    (overview, "left-browser"),
    (detail, "right-browser"),
    height=400,
)


## 6. Opt-in browser stop key

Enable the global browser stop key (default **Ctrl+Q**) to end the whole script
from the browser — like a terminal Ctrl+C. Enable it for the main scene via the
constructor flag or `enable_server_stop_key()`, or per named scene.


In [7]:
viz = Visualizer(enable_server_stop_key=True)  # main scene: Ctrl+Q
overview = viz.scene("overview", enable_server_stop_key=True)

# Or enable afterward with a custom key/modifiers:
# viz.enable_server_stop_key(key="x", modifiers=[KeyModifier.CTRL, KeyModifier.SHIFT])
print("stop key enabled on the main scene and 'overview'")


stop key enabled on the main scene and 'overview'


## Visual Examples

Two named scenes exported as standalone HTML.


In [8]:
viz = Visualizer(reuse_existing=False, title="Tanga — Multi-Scene export")

overview = viz.scene("overview")
detail = viz.scene("detail")

overview.add(Sphere(Point(0, 0, 0), 2.0), color="#4488ff", opacity=0.3, label="$S_1$")
overview.add(Point(1, 1, 1), color="#ff4444", label="$P$")

detail.add(Sphere(Point(2, 1, 0), 1.0), color="#ffcc00", opacity=0.8, label="$S_2$")

overview.display_snapshot()
detail.display_snapshot()


## Summary

| Task | API |
|---|---|
| Named scene | `viz.scene("name")` → `VizSceneHandle` |
| Main scene | the plain `Visualizer` API (scene name `""`) |
| Context manager | `with viz:` / `with viz.scene("name"):` |
| Lifecycle | `add` / `update` / `remove` / `clear(add_axes=, add_grid=)` / `flush` |
| Style without rebuild | `update_style(id, style)` |
| Navigate a browser | `navigate_to(scene, target=...)` |
| Inspect | `list_scenes()` / `list_browsers()` |
| Side-by-side | `display_row((scene, name), ...)` |
| Stop key | `enable_server_stop_key()` (Ctrl+Q) |

**Next:** [07 — Scene Graphs](../07_scene_graphs/).
